In [ ]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover

import pandas as pd
import pandas_ta as ta

from backtesting.test import GOOG

In [2]:
filename = "D:/BTCUSDT-1d.csv"

data = pd.read_csv(filename, parse_dates=True, index_col='timestamp')
data.rename(columns={
    'open': 'Open',
    'high': 'High',
    'low': 'Low',
    'close': 'Close',
    'volume': 'Volume'
}, inplace=True)

In [12]:
class SmaCross(Strategy):
    fast_sma = 20
    slow_sma = 50
    sl = 0.3
    tp = 3
    
    def init(self):
        self.sma_short = self.I(ta.sma, pd.Series(self.data.Close), self.fast_sma)
        self.sma_long = self.I(ta.sma, pd.Series(self.data.Close), self.slow_sma)

    def next(self):  
        enter_price = self.data.Close[-1]

        if crossover(self.sma_short , self.sma_long) and not self.position:
            self.buy(sl = (1. - self.sl) * enter_price, tp = (1.0 + self.tp) * enter_price)
            # self.buy()
        elif crossover(self.sma_long , self.sma_short) and self.position.is_long:
            self.position.close()

In [13]:
bt = Backtest(data, SmaCross, cash=1_000_000, commission=.002, exclusive_orders=True)
stats = bt.run()
print(stats)

Start                     2017-08-17 00:00:00
End                       2025-11-03 00:00:00
Duration                   3000 days 00:00:00
Exposure Time [%]                   50.249917
Equity Final [$]                20497711.1113
Equity Peak [$]                 23209656.2113
Return [%]                        1949.771111
Buy & Hold Return [%]             2387.305721
Return (Ann.) [%]                   44.390174
Volatility (Ann.) [%]               68.396506
Sharpe Ratio                         0.649012
Sortino Ratio                        1.569885
Calmar Ratio                         0.739988
Max. Drawdown [%]                  -59.987723
Avg. Drawdown [%]                  -11.078814
Max. Drawdown Duration     1196 days 00:00:00
Avg. Drawdown Duration       83 days 00:00:00
# Trades                                   33
Win Rate [%]                        39.393939
Best Trade [%]                     299.201597
Worst Trade [%]                    -30.139721
Avg. Trade [%]                    

In [80]:
class SimpleRSI(Strategy):
    length = 14
    overbought = 70
    oversold = 30
    sl = 0.3
    tp = 3
    
    def init(self):
        self.rsi_line = self.I(ta.rsi, pd.Series(self.data.Close), self.length)

    def next(self): 
        enter_price = self.data.Close[-1]
             
        if ( self.oversold > self.rsi_line and not self.position ):
            self.buy(sl = (1. - self.sl) * enter_price, tp = (1.0 + self.tp) * enter_price)
            # self.buy()
        elif (self.overbought < self.rsi_line and self.position.is_long ):
            self.position.close()

In [79]:
bt = Backtest(data, SimpleRSI, cash=1_000_000, commission=.002, exclusive_orders=True)
stats = bt.run()
print(stats)

Start                     2017-08-17 00:00:00
End                       2025-11-03 00:00:00
Duration                   3000 days 00:00:00
Exposure Time [%]                   36.954349
Equity Final [$]                 846488.01548
Equity Peak [$]                  2259927.1977
Return [%]                         -15.351198
Buy & Hold Return [%]             2387.305721
Return (Ann.) [%]                   -2.006606
Volatility (Ann.) [%]               44.793921
Sharpe Ratio                              0.0
Sortino Ratio                             0.0
Calmar Ratio                              0.0
Max. Drawdown [%]                  -81.824585
Avg. Drawdown [%]                  -12.539123
Max. Drawdown Duration     2801 days 00:00:00
Avg. Drawdown Duration      240 days 00:00:00
# Trades                                   22
Win Rate [%]                        45.454545
Best Trade [%]                      74.215585
Worst Trade [%]                    -20.201116
Avg. Trade [%]                    

In [24]:
class SimpleMACDStrategy(Strategy):
    macd_fast = 12
    macd_slow = 26
    signal_smooth = 9
    
    def init(self):
        self.macd_line , self.macd_hist, self.macd_signal = \
            self.I(ta.macd, pd.Series(self.data.Close), self.macd_fast, self.macd_slow, self.signal_smooth)

    def next(self):
        if crossover(self.macd_line , self.macd_signal) and not self.position:
            self.buy()
        elif crossover(self.macd_signal , self.macd_line) and self.position.is_long:
            self.position.close()

In [25]:
bt = Backtest(GOOG, SimpleMACDStrategy, commission=.002, exclusive_orders=True)
stats = bt.run()
print(stats)

Start                     2004-08-19 00:00:00
End                       2013-03-01 00:00:00
Duration                   3116 days 00:00:00
Exposure Time [%]                   56.238361
Equity Final [$]                  32755.27914
Equity Peak [$]                   33241.12914
Return [%]                         227.552791
Buy & Hold Return [%]              703.458242
Return (Ann.) [%]                    14.93492
Volatility (Ann.) [%]               27.073622
Sharpe Ratio                         0.551641
Sortino Ratio                        0.999462
Calmar Ratio                         0.359133
Max. Drawdown [%]                  -41.586044
Avg. Drawdown [%]                   -4.590031
Max. Drawdown Duration      893 days 00:00:00
Avg. Drawdown Duration       57 days 00:00:00
# Trades                                   77
Win Rate [%]                        46.753247
Best Trade [%]                      35.280782
Worst Trade [%]                    -18.874562
Avg. Trade [%]                    